# PT-ROLLOUT-LIFECYCLE. Rollout and Policy Freshness | Rollout 与策略新鲜度

**难度：** Hard | **环境：** CPU-first | **标签：** `后训练`, `Rollout`, `Policy Freshness`

---

## 本节导读

在线后训练不是把生成结果直接送进下一次更新。每条轨迹都由某个策略版本生成，并携带采样状态、奖励状态和失败信息；learner 更新后，旧轨迹与当前策略之间会产生版本差。

本节沿着 `生成 → 奖励 → 准入 → 更新 → 同步` 追踪一条轨迹，输入带身份和奖励状态的轨迹记录，输出可进入当前更新的批次以及可激活的策略版本。

**关键词：** `rollout identity`, `policy version`, `freshness`, `synchronization`

![Rollout 与策略版本生命周期](../docs/public/02_PyTorch_Algorithms/pt_rollout_lifecycle.svg)

## 前置阅读

**导语：** 按准备进入的在线更新分支选择前置页，再观察策略版本变化如何影响轨迹准入。

- PPO / GRPO 分支：[Part 02 · PT-REWARD-SIGNAL 奖励、价值与优势信号](./PT-REWARD-SIGNAL_Reward_Model_Value_and_Advantage.ipynb)
- Online DPO 分支：[Part 02 · 50 偏好数据与对齐评测](./50_Preference_Data_and_Evaluation.ipynb) → [Part 02 · 17 DPO 损失](./17_DPO_Loss_Tutorial.ipynb)

### Step 1：一条 rollout 必须保留哪些身份

轨迹至少要绑定 `rollout_id`、`prompt_id`、`policy_version` 和生成状态。否则 learner 无法判断数据来自哪个策略，也无法把失败、奖励和更新结果追溯到同一次采样。

| 字段 | 回答的问题 | 不满足时的动作 |
| --- | --- | --- |
| `policy_version` | 哪个策略生成了该轨迹 | 缺失则拒绝进入更新批次 |
| `status` | 生成、工具或环境执行是否成功 | 失败轨迹进入失败分析，不伪装成零奖励 |
| `reward_status` | 奖励或 verifier 是否完成 | 未完成则等待或超时，不提前更新 |
| `tokens / logprobs` | 更新所需行为数据是否完整 | 与算法契约不符时拒绝 |

### Step 2：Policy freshness 决定轨迹能否进入当前更新

令当前 learner 版本为 `v`，轨迹生成版本为 `v_r`，则版本滞后为 `lag = v - v_r`。`lag=0` 表示当前策略数据；允许有限异步时可以接受小于阈值的旧轨迹；未来版本、缺失版本或超过阈值的轨迹必须隔离。

| 状态 | 条件 | 处理 |
| --- | --- | --- |
| fresh | `lag == 0` | 可进入 on-policy 更新 |
| tolerated | `0 < lag <= max_lag` | 仅在算法和实验明确允许时接纳 |
| stale | `lag > max_lag` | 拒绝或转入离线缓冲区 |
| future | `lag < 0` | 版本登记错误，立即拒绝 |

### Step 3：更新完成不等于 rollout worker 已切换版本

learner 产出新权重后，还要经过保存、分发、加载和 worker 确认。只有确认完成的 worker 才能声明新版本；更新失败、权重同步失败或 reward 服务超时，都应保留明确状态并阻止错误轨迹进入下一轮。

![策略更新与 worker 同步门槛](../docs/public/02_PyTorch_Algorithms/pt_rollout_version_gate.svg)

### Step 4：实现 rollout 准入与更新批次选择

题目区实现四个连续机制：计算版本新鲜度、校验轨迹状态、形成更新批次，并在所有必需 worker 确认后激活新策略版本。

| TODO / 函数 | 学习者完成的机制 | 必须满足的约束 | 测试证据 |
| --- | --- | --- | --- |
| TODO 1：判断策略新鲜度<br>`classify_freshness` | 按策略版本差判断 rollout 新鲜度 | 未来版本和超过最大滞后的旧版本不能接纳 | 当前版本、允许滞后、过期与未来版本 |
| TODO 2：校验轨迹状态<br>`validate_rollout` | 校验轨迹状态与奖励完成度 | 失败轨迹或奖励未完成不能进入更新 | 成功、执行失败与奖励缺失 |
| TODO 3：形成更新批次<br>`select_update_batch` | 从候选轨迹形成可更新批次 | accepted / rejected 分离并保留拒绝原因 | 混合批次、空批次与多种拒绝原因 |
| TODO 4：激活策略版本<br>`activate_policy_version` | 检查新产物是否已被必需 worker 加载 | 缺失确认或版本不一致时不得声明激活 | 全部确认、缺失 worker 与错误版本 |

In [ ]:
from typing import Any, Dict, List


In [ ]:
def classify_freshness(rollout_version, current_version, max_lag):
    """返回 fresh、tolerated、stale 或 future。"""
    if max_lag < 0:
        raise ValueError('max_lag 必须非负')
    # ==========================================
    # TODO 1：根据版本差判断轨迹新鲜度。
    # lag = ???  # 当前 learner 版本减轨迹生成版本。
    # ==========================================
    raise NotImplementedError("TODO 1：判断策略新鲜度")


def validate_rollout(record):
    """返回 (是否可用, 原因)，不把失败轨迹改写成零奖励。"""
    # ==========================================
    # TODO 2：依次校验身份、生成状态、奖励状态和 Token。
    # required = ???  # 可追溯一条轨迹的必需身份字段。
    # ==========================================
    raise NotImplementedError("TODO 2：校验轨迹状态")


def select_update_batch(records, current_version, max_lag=0):
    """按状态与策略版本筛选更新批次，并保留拒绝原因。"""
    # ==========================================
    # TODO 3：组合轨迹契约与策略新鲜度门槛。
    # accepted = ???  # 可进入当前更新的轨迹。
    # rejected = ???  # 保留轨迹与首要拒绝原因。
    # ==========================================
    raise NotImplementedError("TODO 3：形成更新批次")


def activate_policy_version(artifact_version, worker_versions, required_workers):
    """只有必需 worker 全部确认同一版本时才激活新策略。"""
    # ==========================================
    # TODO 4：检查 worker 确认并返回版本激活状态。
    # missing_workers = ???  # 尚未报告版本的必需 worker。
    # mismatched_workers = ???  # 已报告但版本不一致的 worker。
    # ==========================================
    raise NotImplementedError("TODO 4：激活策略版本")

### 测试

测试覆盖新鲜度边界、失败轨迹、奖励未完成和批次拒绝原因。

In [ ]:
def _record(version=3, status='completed', reward_status='completed', tokens=None):
    return {'rollout_id': 'r-1', 'prompt_id': 'p-1', 'policy_version': version, 'status': status, 'reward_status': reward_status, 'tokens': [1, 2] if tokens is None else tokens}

def test_freshness():
    """版本差必须覆盖 fresh、tolerated、stale 与 future 四种状态。"""
    assert classify_freshness(3, 3, 1) == 'fresh'
    assert classify_freshness(2, 3, 1) == 'tolerated'
    assert classify_freshness(1, 3, 1) == 'stale'
    assert classify_freshness(4, 3, 1) == 'future'

def test_rollout_contract():
    """生成失败或奖励未完成的轨迹不能进入更新。"""
    assert validate_rollout(_record()) == (True, 'ok')
    assert validate_rollout(_record(status='failed'))[0] is False
    assert validate_rollout(_record(reward_status='pending'))[0] is False
    assert validate_rollout(_record(tokens=[])) == (False, 'empty_tokens')

def test_batch_selection():
    """混合批次必须分离 accepted 与带原因的 rejected 轨迹。"""
    records = [_record(3), _record(2), _record(1), _record(3, status='failed')]
    result = select_update_batch(records, current_version=3, max_lag=1)
    assert len(result['accepted']) == 2
    assert {item['reason'] for item in result['rejected']} == {'stale_policy', 'generation_failed'}

def test_version_activation():
    """缺失确认或错误版本不能激活新策略。"""
    workers = {'rollout-0': 4, 'rollout-1': 4, 'reward-0': 4}
    ready = activate_policy_version(4, workers, ['rollout-0', 'rollout-1'])
    assert ready == {'activated': True, 'active_version': 4, 'missing_workers': [], 'mismatched_workers': []}
    blocked = activate_policy_version(4, {'rollout-0': 4, 'rollout-1': 3}, ['rollout-0', 'rollout-1'])
    assert blocked['activated'] is False and blocked['mismatched_workers'] == ['rollout-1']

test_freshness(); test_rollout_contract(); test_batch_selection(); test_version_activation()
print('✅ rollout 生命周期机制测试通过')

---

🛑 **STOP HERE** 🛑

> 请先完成题目区并运行测试，再查看参考实现。

---

## 参考代码与解析

In [ ]:
def classify_freshness(rollout_version, current_version, max_lag):
    if max_lag < 0:
        raise ValueError('max_lag 必须非负')
    lag = current_version - rollout_version
    if lag < 0:
        return 'future'
    if lag == 0:
        return 'fresh'
    return 'tolerated' if lag <= max_lag else 'stale'

def validate_rollout(record):
    required = ('rollout_id', 'prompt_id', 'policy_version')
    if any(key not in record for key in required):
        return False, 'missing_identity'
    if record.get('status') != 'completed':
        return False, 'generation_failed'
    if record.get('reward_status') != 'completed':
        return False, 'reward_not_ready'
    if not record.get('tokens'):
        return False, 'empty_tokens'
    return True, 'ok'

def select_update_batch(records, current_version, max_lag=0):
    accepted, rejected = [], []
    for record in records:
        valid, reason = validate_rollout(record)
        if not valid:
            rejected.append({'record': record, 'reason': reason})
            continue
        freshness = classify_freshness(record['policy_version'], current_version, max_lag)
        if freshness in {'stale', 'future'}:
            rejected.append({'record': record, 'reason': f'{freshness}_policy'})
        else:
            accepted.append(record | {'freshness': freshness})
    return {'accepted': accepted, 'rejected': rejected}

def activate_policy_version(artifact_version, worker_versions, required_workers):
    # TODO 4：激活策略版本。
    missing_workers = sorted(worker for worker in required_workers if worker not in worker_versions)
    mismatched_workers = sorted(worker for worker in required_workers
                                if worker in worker_versions and worker_versions[worker] != artifact_version)
    activated = not missing_workers and not mismatched_workers
    return {'activated': activated, 'active_version': artifact_version if activated else None,
            'missing_workers': missing_workers, 'mismatched_workers': mismatched_workers}

### 解析

#### TODO 1：判断策略新鲜度

- 使用 learner 与 rollout 的版本差区分 fresh、tolerated、stale 和 future。

#### TODO 2：校验轨迹状态

- 生成失败、奖励未完成和空 token 分别返回原因，不改写成零奖励。

#### TODO 3：形成更新批次

- 组合状态与版本准入，可用轨迹与拒绝轨迹都保留可追溯结果。

#### TODO 4：激活策略版本

- 新产物存在不代表版本已生效；必需 worker 全部确认同一版本后，下一轮采样才能声明新版本。

## 相关阅读

- [Part 02 · PT-RL-SYSTEM 采样、训练与奖励系统](./PT-RL-SYSTEM_Rollout_Training_and_Reward_System.ipynb)
- [Part 02 · 51 Online DPO](./51_Online_DPO.ipynb)
- [TRL 文档](https://huggingface.co/docs/trl/)
- [verl](https://github.com/volcengine/verl)
- [OpenRLHF](https://github.com/OpenRLHF/OpenRLHF)